## Configuration ##

In [50]:
import os
import json
import pathlib
import requests
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from dbrepo.RestClient import RestClient
load_dotenv()

USERNAME = os.getenv("DBREPO_USER")
PASSWORD = os.getenv("DBREPO_PASSWORD")

auth = (USERNAME, PASSWORD)
HOST = "https://test.dbrepo.tuwien.ac.at"
PORT = 3306

API_BASE  = 'https://test.dbrepo.tuwien.ac.at/api/v1'
DATABASE = "data_stewardship_group6_crash_serverity_prediction_lkhb"
DB_ID = "3d81c073-e5fd-49b9-9536-b75ed490ca3e"


print('Configuration loaded.')

Configuration loaded.


In [62]:
# --- collision ---
collision_df = pd.DataFrame({
    "collision_index": pd.Series(dtype="object"),
    "collision_year": pd.Series(dtype="Int64"),
    "collision_ref_no": pd.Series(dtype="object"),
    "location_easting_osgr": pd.Series(dtype="Int64"),
    "location_northing_osgr": pd.Series(dtype="Int64"),
    "longitude": pd.Series(dtype="float64"),
    "latitude": pd.Series(dtype="float64"),
    "police_force": pd.Series(dtype="object"),
    "collision_severity": pd.Series(dtype="object"),
    "number_of_vehicles": pd.Series(dtype="Int64"),
    "number_of_casualties": pd.Series(dtype="Int64"),
    "date": pd.Series(dtype="datetime64[ns]"),
    "day_of_week": pd.Series(dtype="object"),
    "time": pd.Series(dtype="object"),
    "local_authority_district": pd.Series(dtype="object"),
    "local_authority_ons_district": pd.Series(dtype="object"),
    "local_authority_highway": pd.Series(dtype="object"),
    "local_authority_highway_current": pd.Series(dtype="object"),
    "first_road_class": pd.Series(dtype="object"),
    "first_road_number": pd.Series(dtype="Int64"),
    "road_type": pd.Series(dtype="object"),
    "speed_limit": pd.Series(dtype="Int64"),
    "junction_detail": pd.Series(dtype="object"),
    "junction_control": pd.Series(dtype="object"),
    "second_road_class": pd.Series(dtype="object"),
    "second_road_number": pd.Series(dtype="Int64"),
    "pedestrian_crossing": pd.Series(dtype="object"),
    "light_conditions": pd.Series(dtype="object"),
    "weather_conditions": pd.Series(dtype="object"),
    "road_surface_conditions": pd.Series(dtype="object"),
    "special_conditions_at_site": pd.Series(dtype="object"),
    "carriageway_hazards": pd.Series(dtype="object"),
    "urban_or_rural_area": pd.Series(dtype="object"),
    "did_police_officer_attend_scene_of_accident": pd.Series(dtype="boolean"),
    "trunk_road_flag": pd.Series(dtype="boolean"),
    "lsoa_of_accident_location": pd.Series(dtype="object"),
    "enhanced_severity_collision": pd.Series(dtype="object"),
    "collision_injury_based": pd.Series(dtype="boolean"),
    "collision_adjusted_severity_serious": pd.Series(dtype="boolean"),
    "collision_adjusted_severity_slight": pd.Series(dtype="boolean"),
})
# --- vehicle ---
vehicle_df = pd.DataFrame({
    "vehicle_id": pd.Series(dtype="int64"),
    "collision_index": pd.Series(dtype="object"),
    "vehicle_reference": pd.Series(dtype="int64"),
    "vehicle_type": pd.Series(dtype="object"),
    "towing_and_articulation": pd.Series(dtype="object"),
    "vehicle_manoeuvre": pd.Series(dtype="object"),
    "vehicle_direction_from": pd.Series(dtype="object"),
    "vehicle_direction_to": pd.Series(dtype="object"),
    "vehicle_location_restricted_lane": pd.Series(dtype="object"),
    "junction_location": pd.Series(dtype="object"),
    "skidding_and_overturning": pd.Series(dtype="object"),
    "hit_object_in_carriageway": pd.Series(dtype="object"),
    "vehicle_leaving_carriageway": pd.Series(dtype="object"),
    "hit_object_off_carriageway": pd.Series(dtype="object"),
    "first_point_of_impact": pd.Series(dtype="object"),
    "vehicle_left_hand_drive": pd.Series(dtype="bool"),
    "journey_purpose_of_driver": pd.Series(dtype="object"),
    "sex_of_driver": pd.Series(dtype="object"),
    "age_of_driver": pd.Series(dtype="int64"),
    "age_band_of_driver": pd.Series(dtype="object"),
    "engine_capacity_cc": pd.Series(dtype="int64"),
    "propulsion_code": pd.Series(dtype="object"),
    "age_of_vehicle": pd.Series(dtype="int64"),
    "generic_make_model": pd.Series(dtype="object"),
    "driver_imd_decile": pd.Series(dtype="int64"),
    "lsoa_of_driver": pd.Series(dtype="object"),
    "escooter_flag": pd.Series(dtype="bool"),
    "driver_distance_banding": pd.Series(dtype="object"),
})

# --- casualty ---
casualty_df = pd.DataFrame({
    "casualty_id": pd.Series(dtype="int64"),
    "collision_index": pd.Series(dtype="object"),
    "vehicle_reference": pd.Series(dtype="int64"),
    "casualty_reference": pd.Series(dtype="int64"),
    "casualty_class": pd.Series(dtype="object"),
    "sex_of_casualty": pd.Series(dtype="object"),
    "age_of_casualty": pd.Series(dtype="int64"),
    "age_band_of_casualty": pd.Series(dtype="object"),
    "casualty_severity": pd.Series(dtype="object"),
    "pedestrian_location": pd.Series(dtype="object"),
    "pedestrian_movement": pd.Series(dtype="object"),
    "car_passenger": pd.Series(dtype="object"),
    "bus_or_coach_passenger": pd.Series(dtype="object"),
    "pedestrian_road_maintenance_worker": pd.Series(dtype="object"),
    "casualty_type": pd.Series(dtype="object"),
    "casualty_imd_decile": pd.Series(dtype="int64"),
    "lsoa_of_casualty": pd.Series(dtype="object"),
    "enhanced_casualty_severity": pd.Series(dtype="object"),
    "casualty_injury_based": pd.Series(dtype="bool"),
    "casualty_adjusted_severity_serious": pd.Series(dtype="bool"),
    "casualty_adjusted_severity_slight": pd.Series(dtype="bool"),
    "casualty_distance_banding": pd.Series(dtype="object"),
})


tables = [
    {
        "name":             "collision",
        "description":      (
            "One row per road collision reported to the police in Great Britain during 2023. "
            "Contains geospatial coordinates, date/time, vehicle and casualty counts, "
            "road characteristics, environmental conditions, and administrative attributes."
        ),
        "dataframe":        collision.set_index("collision_index"),
        "is_public":        True,
        "is_schema_public": True,
    },
    {
        "name":             "vehicle",
        "description":      (
            "One row per vehicle involved in a recorded collision. "
            "Stores vehicle characteristics, manoeuvre details, driver demographics, "
            "propulsion type, and journey purpose. References collision via collision_index (FK)."
        ),
        "dataframe":        vehicle.set_index("vehicle_id"),
        "is_public":        True,
        "is_schema_public": True,
    },
    {
        "name":             "casualty",
        "description":      (
            "One row per casualty in a recorded collision. "
            "Captures casualty class, severity, demographics, pedestrian details, "
            "and adjusted severity flags. References collision via collision_index (FK)."
        ),
        "dataframe":        casualty.set_index("casualty_id"),
        "is_public":        True,
        "is_schema_public": True,
    },
]


print('Jsons + DFs defined:')

Jsons + DFs defined:


In [63]:
from dbrepo.RestClient import RestClient
url = f"{API_BASE}/database/{DB_ID}"

client =RestClient(endpoint=HOST, username=USERNAME, password=PASSWORD, secure=True)


for t in tables:
    print(f"Creating table '{t['name']}, end=" ")
    result = client.create_table(
        database_id=DB_ID,
        name=t["name"],
        is_public=t["is_public"],
        is_schema_public=t["is_schema_public"],
        dataframe=t["dataframe"],
        description=t["description"],
        with_data=False
    )
    print
    print(f"done (table_id={result.id})")

Creating table 'collision'... 2026-05-22 18:08:29,669 root         WARNING default to 'text' for column date and type <class 'numpy.dtype'>


MalformedError: Failed to create table: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Validation failure","instance":"/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/table","properties":null}

## Create citable identifier ##

In [70]:
payload = {
        'databaseId':      DB_ID,
        'type':            'DATABASE',
        'publicationYear': 2023,
        'titles': [
            {'title': 'UK Road Safety Open Data 2023', 'titleType': 'MAIN'}
        ],
        'descriptions': [
            {
                'description': (
                    'Road safety and traffic collision data for Great Britain for the year 2023, '
                    'originally published by the UK Department for Transport under the '
                    'Open Government Licence v3.0. '
                    'Covers reported accidents, involved vehicles, casualties, '
                    'and associated road and environmental conditions. '
                    'This 3NF relational database was created for academic purposes '
                    'as part of the Data Stewardship course at TU Wien (Group 6, 2026).'
                ),
                'descriptionType': 'ABSTRACT'
            }
        ],
        'creators': [
            {'creatorName': 'Department for Transport, United Kingdom',
             'nameType':    'ORGANISATIONAL'}
        ],
        'licenses': [
            {'identifier': 'OGL-UK-3.0',
             'uri': 'https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/'}
        ],
        'relatedIdentifiers': [
            {'relatedIdentifier':     'https://www.gov.uk/government/statistical-data-sets/road-safety-open-data',
             'relatedIdentifierType': 'URL',
             'relationType':          'IS_DERIVED_FROM'}
        ],
        'funders': [
            {'funderName': 'Crown Copyright – Department for Transport, United Kingdom'}
        ]
    }


ValidationError: 3 validation errors for CreateIdentifier
creators.0.name_type
  Input should be 'Personal' or 'Organizational' [type=enum, input_value='organisational', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/enum
related_identifiers.0.type
  Input should be 'DOI', 'URL', 'URN', 'ARK', 'arXiv', 'bibcode', 'EAN13', 'EISSN', 'Handle', 'IGSN', 'ISBN', 'ISTC', 'LISSN', 'LSID', 'PMID', 'PURL', 'UPC' or 'w3id' [type=enum, input_value='url', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/enum
related_identifiers.0.relation
  Input should be 'IsCitedBy', 'Cites', 'IsSupplementTo', 'IsSupplementedBy', 'IsContinuedBy', 'Continues', 'IsDescribedBy', 'Describes', 'HasMetadata', 'IsMetadataFor', 'HasVersion', 'IsVersionOf', 'IsNewVersionOf', 'IsPreviousVersionOf', 'IsPartOf', 'HasPart', 'IsPublishedIn', 'IsReferencedBy', 'References', 'IsDocumentedBy', 'Documents', 'IsCompiledBy', 'Compiles', 'IsVariantFormOf', 'IsOri